In [0]:
!pip install -U datasets "fsspec>=2024.2.0"

In [0]:
dbutils.library.restartPython()

In [0]:
import huggingface_hub, datasets, fsspec
print("huggingface_hub:", huggingface_hub.__version__)
print("datasets:", datasets.__version__)
print("fsspec:", fsspec.__version__)

# huggingface_hub: 1.3.5
# datasets: 2.18.0
# fsspec: 2023.5.0

In [0]:
import os
import json

from datasets import load_dataset

import mlflow
from mlflow.genai.scorers import scorer
from mlflow.entities import AssessmentSource, Feedback

from openai import OpenAI

from utils import (
    generate_urls,
    calculate_invoice_accuracies,
    calculate_key_level_metrics,
    calculate_individual_invoice_accuracies,
    convert_base64_to_pil,
    retrieve_token_usage,
)

from prompt import (
    register_prompt,
    set_prompt_alias,
    get_prompt
)

In [0]:
mlflow.get_registry_uri()

In [0]:
mlflow.get_tracking_uri()

In [0]:
MODEL_NAME = 'databricks-gpt-5-1' # databricks-gpt-5-2
REASONING = 'low'
MLFLOW_EXPERIMENT_NAME = '/Workspace/Users/biswadeep.upadhyay@databricks.com/mlflow_artifacts/invoice-extraction-gpt5.1-baseline'
PROMPT_NAME = 'invoice-extraction-gpt5-1-prompt'
PROMPT_VERSION = '1'
UC_CATALOG = 'mlops_beepz_dev'
UC_SCHEMA = 'invoice_extraction'
prompt_name = f"{UC_CATALOG}.{UC_SCHEMA}.{PROMPT_NAME}"

In [0]:
# auth for language model
# base_url = f'https://{spark.conf.get("spark.databricks.workspaceUrl")}/serving-endpoints'
# databricks_token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

mlflow_creds = mlflow.utils.databricks_utils.get_databricks_host_creds()

# openai client
client = OpenAI(
    api_key=mlflow_creds.token,
    base_url=f"{mlflow_creds.host}/serving-endpoints"
  )

In [0]:
# enable mlflow tracing
mlflow.openai.autolog()

In [0]:
# set up mlflow experiment for logging all the traces of this notebook
experiment = mlflow.set_experiment(experiment_name=MLFLOW_EXPERIMENT_NAME)
mlflow.set_experiment_tags({"owner": "beepz"})

In [0]:
experiment.experiment_id

In [0]:
# prompt registry

In [0]:
# Link the current MLflow experiment to a UC schema for prompts
mlflow.set_experiment_tags({
    "mlflow.promptRegistryLocation": f"{UC_CATALOG}.{UC_SCHEMA}"
})

In [0]:
system_prompt = """You are a Vision Language Model designed to extract structured data from invoice receipts.
Task:
Convert the invoice receipt into a well-formed JSON object strictly following the schema provided.

Requirements:
1. Identify and extract only these sections (if present): `menu`, `sub_menu`, `sub_total`, `total`.  
2. Preserve exact formatting for all the extracted values.  
3. Do not output fields that lack data—omit empty keys.  
4. Do not add any information not present in the invoice.
5. In case of prices and currencies, ensure to maintain the original format without any modifications.

Schema:
{{schema}}

Output:
Return valid, minimal JSON matching this schema - no extraneous keys or null values.
"""

In [0]:
#register_prompt(prompt_name, system_prompt)

In [0]:
prompt = get_prompt(prompt_name=prompt_name, version=PROMPT_VERSION)

In [0]:
prompt

In [0]:
# LOAD DATA
dataset = load_dataset("naver-clova-ix/cord-v2")
dataset

In [0]:
dataset["test"][0]["image"]

In [0]:
dataset["test"][0]['ground_truth']

In [0]:
example_1 = json.loads(dataset["validation"][0]["ground_truth"])["gt_parse"]
example_2 = json.loads(dataset["validation"][1]["ground_truth"])["gt_parse"]
example_3 = json.loads(dataset["validation"][2]["ground_truth"])["gt_parse"]

In [0]:
example_1

In [0]:
# DATA PREPARATION
with open("schema.json", "r") as f:
    schema_dict = json.load(f)

schema_dict

In [0]:
NUM_SAMPLES = 10
test_dataset = dataset["test"].select(range(NUM_SAMPLES))
url_list, ground_truth_list = generate_urls(dataset=test_dataset)

In [0]:
ground_truth_list

In [0]:
# inference and evaluation
import os

if not os.path.exists('artifacts'):
  os.makedirs('artifacts')

eval_dataset = []
for index, url in enumerate(url_list):
  eval_dict = {
    "inputs" : {"image_base64": url, "schema": schema_dict},
    "expectations" : ground_truth_list[index]
  }

  eval_dataset.append(eval_dict)


eval_dataset # testing the value

     

In [0]:
def predict_fn(image_base64, schema) -> str:
  system_prompt_template = mlflow.genai.load_prompt(
        name_or_uri=PROMPT_NAME, version=PROMPT_VERSION
    )
  if "fewshot" in PROMPT_NAME:
      system_prompt = system_prompt_template.format(
          schema=schema, example1=example_1, example2=example_2, example3=example_3
      )
  else:
      system_prompt = system_prompt_template.format(schema=schema)


  response = client.responses.create(
        model=MODEL_NAME,
        reasoning={
            "effort": REASONING,
        },
        text={"verbosity": "low"},
        input=[
            {
                "role": "user",
                "content": [
                    {"type": "input_text", "text": system_prompt},
                    {"type": "input_image", "image_url": f"data:image/jpeg;base64,{image_base64}"},
                ],
            }
        ],
    )
  response_text = response.output[1].content[0].text

  return response_text

In [0]:
@scorer
def exact_match(inputs, outputs, expectations, trace) -> Feedback:
    outputs = json.loads(outputs)
    expectations = expectations["expected_response"]
    trace_id = trace.info.trace_id

    # Create child run for every invoice
    with mlflow.start_run(parent_run_id=parent_run.info.run_id, nested=True, run_name=trace_id):
        # Log the prediction and ground truth
        mlflow.log_dict(outputs, "prediction.json")
        mlflow.log_dict(expectations, "ground_truth.json")

        # Compute accuracy @ invoice level
        pred_df, acc = calculate_individual_invoice_accuracies(
            ground_truth=expectations, output=outputs
        )
        mlflow.log_param("trace_id", trace_id)
        mlflow.log_metric("accuracy", acc)

        # Log predictions as artifacts
        pred_df.to_csv(f"artifacts/predictions_{trace_id}.csv", index=False)
        mlflow.log_artifact(local_path=f"artifacts/predictions_{trace_id}.csv")

        # Save the invoice image
        base64_string = inputs["image_base64"]
        image = convert_base64_to_pil(base64_string)
        mlflow.log_image(image, f"input_image_{trace_id}.png")

    return Feedback(value=round(acc, 2), name="Accuracy")

In [0]:
mlflow.end_run()

In [0]:
import time

parent_run = mlflow.start_run(run_name=f"{MODEL_NAME}-{REASONING}-evaluation")

start_time = time.time()

results = mlflow.genai.evaluate(
    data=eval_dataset,
    scorers=[exact_match],
    predict_fn=predict_fn,
)

total_time = time.time() - start_time

In [0]:
testing the 